# Object Tracking

This notebook extends the YOLO video detection pipeline
with object tracking.

## Objectives

- Understand the difference between detection and tracking
- Match detections across consecutive frames
- Assign persistent track IDs
- Build a simple IoU-based tracker
- Apply a real multi-object tracker
- Visualize object identities and trajectories

This is the third stage of the
**Real-Time Object Detection & Tracking** project.

## 1. Detection vs Tracking

Object Detection answers:

> What objects are present in this frame, and where are they?

Object Tracking answers:

> Which detected object in the current frame
> corresponds to an object from a previous frame?

Example:

Frame 1:
- Person
- Person
- Car

Frame 2:
- Person
- Person
- Car

Detection treats every frame independently.

Tracking attempts to maintain identity:

Frame 1:
- Person #1
- Person #2
- Car #3

Frame 2:
- Person #1
- Person #2
- Car #3

In [1]:
def calculate_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])

    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection_area = (
        intersection_width * intersection_height
    )

    area_a = (
        (box_a[2] - box_a[0]) *
        (box_a[3] - box_a[1])
    )

    area_b = (
        (box_b[2] - box_b[0]) *
        (box_b[3] - box_b[1])
    )

    union_area = area_a + area_b - intersection_area

    if union_area == 0:
        return 0.0

    return intersection_area / union_area

In [2]:
previous_box = [100, 100, 150, 250]
current_box = [105, 102, 155, 252]

iou = calculate_iou(previous_box, current_box)

print(f"IoU: {iou:.4f}")

IoU: 0.7986


In [3]:
previous_boxes = [
    [100, 100, 150, 250],   # Person #1
    [300, 100, 350, 250],   # Person #2
]

current_boxes = [
    [105, 102, 155, 252],   # likely #1
    [305, 98, 355, 248],    # likely #2
]

In [4]:
def match_detections(previous_boxes, current_boxes, iou_threshold=0.3):
    matches = []

    used_previous = set()

    for current_index, current_box in enumerate(current_boxes):

        best_iou = 0
        best_previous_index = None

        for previous_index, previous_box in enumerate(previous_boxes):

            if previous_index in used_previous:
                continue

            iou = calculate_iou(
                current_box,
                previous_box
            )

            if iou > best_iou:
                best_iou = iou
                best_previous_index = previous_index

        if best_iou >= iou_threshold:
            matches.append(
                (
                    best_previous_index,
                    current_index,
                    best_iou
                )
            )

            used_previous.add(best_previous_index)

    return matches

In [5]:
matches = match_detections(
    previous_boxes,
    current_boxes
)

for previous_index, current_index, iou in matches:
    print(
        f"Previous #{previous_index + 1} "
        f"→ Current #{current_index + 1} "
        f"(IoU={iou:.3f})"
    )

Previous #1 → Current #1 (IoU=0.799)
Previous #2 → Current #2 (IoU=0.799)


## 4. From Matching to Track IDs

Matching tells us which detection in the current frame
corresponds to a detection from the previous frame.

However, a tracker needs something more:

**Persistent identity.**

Instead of only saying:

Frame 1:
- Detection A
- Detection B

Frame 2:
- Detection A
- Detection B

we want:

Frame 1:
- Person #1
- Person #2

Frame 2:
- Person #1
- Person #2

The ID should remain the same while the tracker
believes that the object is the same.

### Simple IoU-Based Tracker

We will now build a minimal tracker to demonstrate
the basic tracking lifecycle.

The tracker will:

1. Compare current detections with existing tracks.
2. Match detections using IoU.
3. Keep the existing track ID for matched objects.
4. Create a new ID for unmatched detections.

This is an educational implementation.

It is not intended to replace a production tracker
such as ByteTrack.

In [6]:
class SimpleTracker:

    def __init__(self, iou_threshold=0.3):
        self.iou_threshold = iou_threshold
        self.next_id = 1
        self.tracks = {}

    def update(self, detections):

        updated_tracks = {}

        for detection in detections:

            best_iou = 0
            best_track_id = None

            for track_id, previous_box in self.tracks.items():

                iou = calculate_iou(
                    previous_box,
                    detection
                )

                if iou > best_iou:
                    best_iou = iou
                    best_track_id = track_id

            if best_iou >= self.iou_threshold:
                track_id = best_track_id
            else:
                track_id = self.next_id
                self.next_id += 1

            updated_tracks[track_id] = detection

        self.tracks = updated_tracks

        return self.tracks

## 5. Testing the Simple Tracker

Before connecting the tracker to YOLO,
we will test it with synthetic detections.

The object positions will change slightly
between frames.

The expected behavior is that the same objects
keep their track IDs.

In [7]:
tracker = SimpleTracker(iou_threshold=0.3)

frames = [
    [
        [100, 100, 150, 250],
        [300, 100, 350, 250],
    ],

    [
        [105, 102, 155, 252],
        [305, 98, 355, 248],
    ],

    [
        [110, 105, 160, 255],
        [310, 100, 360, 250],
    ],
]

for frame_number, detections in enumerate(frames, start=1):

    tracks = tracker.update(detections)

    print(f"\nFrame {frame_number}")

    for track_id, box in tracks.items():
        print(
            f"Track ID: {track_id}, "
            f"Box: {box}"
        )


Frame 1
Track ID: 1, Box: [100, 100, 150, 250]
Track ID: 2, Box: [300, 100, 350, 250]

Frame 2
Track ID: 1, Box: [105, 102, 155, 252]
Track ID: 2, Box: [305, 98, 355, 248]

Frame 3
Track ID: 1, Box: [110, 105, 160, 255]
Track ID: 2, Box: [310, 100, 360, 250]


## 6. New Object Appearing

A real video does not contain a fixed number of objects.

Objects can enter the scene at any time.

We will now introduce a new detection
in the third frame.

The tracker should:

- Preserve the IDs of existing objects.
- Assign a new ID to the new object.

In [8]:
tracker = SimpleTracker(iou_threshold=0.3)

frames = [
    [
        [100, 100, 150, 250],
        [300, 100, 350, 250],
    ],

    [
        [105, 102, 155, 252],
        [305, 98, 355, 248],
    ],

    [
        [110, 105, 160, 255],
        [310, 100, 360, 250],
        [500, 120, 550, 260],
    ],
]

for frame_number, detections in enumerate(frames, start=1):

    tracks = tracker.update(detections)

    print(f"\nFrame {frame_number}")

    for track_id, box in tracks.items():
        print(
            f"Track ID: {track_id}, "
            f"Box: {box}"
        )


Frame 1
Track ID: 1, Box: [100, 100, 150, 250]
Track ID: 2, Box: [300, 100, 350, 250]

Frame 2
Track ID: 1, Box: [105, 102, 155, 252]
Track ID: 2, Box: [305, 98, 355, 248]

Frame 3
Track ID: 1, Box: [110, 105, 160, 255]
Track ID: 2, Box: [310, 100, 360, 250]
Track ID: 3, Box: [500, 120, 550, 260]


## 7. Limitations of the Simple Tracker

The simple IoU-based tracker demonstrates the core idea
of maintaining object identities.

However, it has important limitations.

### 1. Fast movement

If an object moves significantly between frames,
the IoU may become too small.

### 2. Occlusion

An object may temporarily disappear behind another object.

### 3. Missed detections

The detector may fail to detect an object in one frame.

### 4. Multiple nearby objects

Two objects of the same class may overlap,
making simple spatial matching ambiguous.

### 5. Track management

A real tracker needs rules for:

- Creating tracks
- Updating tracks
- Temporarily losing tracks
- Removing old tracks
- Matching detections robustly

Therefore, a production tracking algorithm
needs more than simple IoU matching.

## 8. ByteTrack

ByteTrack is a multi-object tracking algorithm
designed to associate object detections across frames.

A key idea is that tracking should not rely only
on high-confidence detections.

Lower-confidence detections can also contain useful
information for maintaining object identities,
especially when an object is partially occluded.

In our project, the pipeline becomes:

Video
  ↓
YOLO Detection
  ↓
ByteTrack
  ↓
Track IDs
  ↓
Visualization

In [9]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.9 MB/s eta 0:00:00


In [10]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [11]:
video_path="/content/15956736_640_360_60fps.mp4"

results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    save=True,
    verbose=False
)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 501ms
Prepared 1 package in 136ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to /content/runs/detect/track
